# Baselines

## Imports & Loads

In [29]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, median_absolute_error, r2_score

df_train = pd.read_csv("../data/work/train.csv")
df_test = pd.read_csv("../data/work/test.csv")
y_train = df_train["total"]
y_test = df_test["total"]
x_test_district = df_test["district"]
x_train_district = df_train["district"]

## Global Median Baseline

In [30]:
y_train_median = float(np.median(y_train))
y_pred_median = np.full_like(y_test, fill_value=y_train_median, dtype=float)

mae   = mean_absolute_error(y_test, y_pred_median)
medae = median_absolute_error(y_test, y_pred_median)
r2    = r2_score(y_test, y_pred_median)

print(f"Baseline (Global Median) -> MAE: {mae:.2f}  |  MedAE: {medae:.2f}  |  R²: {r2:.4f}")

Baseline (Global Median) -> MAE: 2135.46  |  MedAE: 1249.50  |  R²: -0.0934


# District Baseline

In [31]:
x_test_district = x_test_district.astype(str).str.strip().str.lower()
x_train_district = x_train_district.astype(str).str.strip().str.lower()

mean_by_district_train = df_train.groupby("district")["total"].mean()
district_pred_mean = df_test["district"].map(mean_by_district_train).fillna(y_train_median).astype(float)

median_by_district_train = df_train.groupby("district")["total"].median()
y_pred_district = df_test["district"].map(mean_by_district_train).fillna(y_train_median).astype(float)

fallback_n   = (~df_test["district"].isin(mean_by_district_train.index)).sum()
fallback_pct = 100 * fallback_n / len(df_test)

mae_district   = mean_absolute_error(y_test, y_pred_district)
medae_district = median_absolute_error(y_test, y_pred_district)
r2_district    = r2_score(y_test, y_pred_district)

print(f"[Baseline] District Mean (+fallback) -> MAE: {mae_district:.2f} | MedAE: {medae_district:.2f} | R²: {r2_district:.4f}")
print(f"Fallback used: {fallback_n} rows ({fallback_pct:.2f}%)")

[Baseline] District Mean (+fallback) -> MAE: 1857.19 | MedAE: 1205.82 | R²: 0.3105
Fallback used: 82 rows (3.52%)


## Distric + Type Baseline

In [32]:
for c in ["district", "type"]:
    df_train[c] = df_train[c].astype("string").str.strip().str.lower()
    df_test[c]  = df_test[c].astype("string").str.strip().str.lower()

median_by_dt = (
    df_train
    .groupby(["district", "type"], dropna=False)["total"]
    .median()
)

district_test = pd.MultiIndex.from_frame(df_test[["district", "type"]])

y_pred_dt = median_by_dt.reindex(district_test)


fallback_mask = y_pred_dt.isna()
fallback_n = int(fallback_mask.sum())
fallback_pct = 100.0 * fallback_n / len(df_test)


y_train_median = df_train["total"].median()
y_pred_dt = y_pred_dt.fillna(y_train_median).astype(float).to_numpy()

mae_district_type   = mean_absolute_error(y_test, y_pred_dt)
medae_district_type = median_absolute_error(y_test, y_pred_dt)
r2_district_type    = r2_score(y_test, y_pred_dt)

print(f"[Baseline] District+Type Median (+fallback) -> MAE: {mae_district_type:.2f} | "
      f"MedAE: {medae_district_type:.2f} | R²: {r2_district_type:.4f}")
print(f"Fallback used: {fallback_n} rows ({fallback_pct:.2f}%)")


[Baseline] District+Type Median (+fallback) -> MAE: 1695.54 | MedAE: 958.75 | R²: 0.3174
Fallback used: 219 rows (9.39%)


## Percentage change in the MAE

In [33]:
pc_mae_district = (mae - mae_district ) / mae * 100
pc_mae_district_type = (mae - mae_district_type ) / mae * 100

print(f"District Baseline: {pc_mae_district}")
print(f"District + Type Baseline: {pc_mae_district_type}")

District Baseline: 13.031070510930542
District + Type Baseline: 20.60102612502259
